# Naive Bayes: Sentiment Analysis

## Step 1: Loading the dataset

In [ ]:
import pandas as pd

total_data = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv")

total_data.head()

## Step 2: Study of variables and their content

### Removing spaces and converting the text to lowercase

In [ ]:
def apply_preprocess(df):
    df = df.drop("package_name", axis=1)
    df["review"] = df["review"].str.strip().str.lower()

    return df

total_data = apply_preprocess(total_data)

total_data.head()

### Divide the dataset into train and test

In [ ]:
from sklearn.model_selection import train_test_split

X = total_data["review"]
y = total_data["polarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

### Transform the text into a word count matrix

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec_model = CountVectorizer(stop_words = "english")
X_train = vec_model.fit_transform(X_train).toarray()
X_test = vec_model.transform(X_test).toarray()

X_train

## Step 3: Build a naive bayes model

I select the MultinomialNB because just the target is binary while the predictors are categorical numbers.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

I will test the other sklearn Naive Bayes models:

In [ ]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB

for model_aux in [GaussianNB(), BernoulliNB()]:
    model_aux.fit(X_train, y_train)
    y_pred_aux = model_aux.predict(X_test)
    print(f"{model_aux} with accuracy: {accuracy_score(y_test, y_pred_aux)}")

I can confirm that the best model is the one I have chosen based on its theoretical foundation.

## Step 4: Optimize the previous model

In [ ]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

hyperparams = {
    "alpha": np.linspace(0.01, 10.0, 200),
    "fit_prior": [True, False]
}

# We initialize the random search
random_search = RandomizedSearchCV(model, hyperparams, n_iter = 50, scoring = "accuracy", cv = 5, random_state = 42)
random_search

In [ ]:
random_search.fit(X_train, y_train)

print(f"Best hyperparameters: {random_search.best_params_}")

After identifying the best hyperparameters, we re-trained the model

In [ ]:
model = MultinomialNB(alpha = 1.917638190954774, fit_prior = False)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

We have improved the model!

## Step 5: Save the model

In [ ]:
from pathlib import Path
from pickle import dump

Path("../models").mkdir(parents=True, exist_ok=True)
dump(model, open("../models/naive_bayes_alpha_1-9176382_fit_prior_False_42.sav", "wb"))

## Step 6: Explore other alternatives

Trying a random Forest Next:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators = 60, random_state = 42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

In [ ]:
from pickle import dump

dump(model, open("../models/ranfor_classifier_nestimators-60_42.sav", "wb"))

As we can see, using a Random Forest produces about an 80% accuracy score.